<a href="https://colab.research.google.com/github/bhurlasravan-creator/capstion-project/blob/main/Module_2_%E2%80%94_Analytics_Pipeline_(_analytics).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)

# Optional SMOTE import with fallback handling if imblearn is missing
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

warnings.filterwarnings("ignore")

# Set global plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

try:
    ANALYTICS_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback when running inside Jupyter Notebook or IPython interactive environment
    ANALYTICS_DIR = os.getcwd()

CSV_PATH = os.path.join(ANALYTICS_DIR, "titanic.csv")
MODEL_PATH = os.path.join(ANALYTICS_DIR, "best_titanic_pipeline.joblib")

# ==========================================
# TASK 1 & PART A: PROFILING AND DATA LOAD
# ==========================================

def load_and_profile_data() -> pd.DataFrame:
    """Loads titanic dataset (using seaborn or offline csv fallback) exactly once."""
    print("=" * 80)
    print("TASK 1: DATASET LOADING AND PROFILING")
    print("=" * 80)

    if os.path.exists(CSV_PATH):
        print(f"Loading dataset from local offline fallback: '{CSV_PATH}'...")
        df = pd.read_csv(CSV_PATH)
    else:
        print("Fetching dataset via seaborn loader 'sns.load_dataset(\"titanic\")'...")
        df = sns.load_dataset("titanic")
        # Immediately save local fallback for offline grading
        df.to_csv(CSV_PATH, index=False)
        print(f"Saved local offline copy to '{CSV_PATH}'.")

    print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("\nDataFrame Info:")
    df.info()

    print("\nNumerical Summary Statistics:")
    print(df.describe())

    print("\nMissing Value Percentage per Column:")
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({"Missing Count": df.isnull().sum(), "Missing Pct (%)": missing_pct})
    missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values(by="Missing Pct (%)", ascending=False)
    print(missing_df.to_string())

    return df

# ==========================================
# TASK 2: MISSING VALUE HANDLING STRATEGY
# ==========================================

def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies strict threshold-based missing value handling strategy:
    - <5% missing: drop rows (embarked, embark_town)
    - 5%-30% missing: impute median (age)
    - >30% missing: drop column (deck ~77.1%)
    """
    print("\n" + "=" * 80)
    print("TASK 2: MISSING VALUE CLEANING STRATEGY")
    print("=" * 80)

    df_clean = df.copy()

    # Deck: 77.1% missing (>30% threshold -> drop column)
    if "deck" in df_clean.columns:
        print("• 'deck' (77.10% missing > 30%): Dropping column due to excessive missingness.")
        df_clean = df_clean.drop(columns=["deck"])

    # Embarked / Embark_town: 0.22% missing (<5% threshold -> drop rows)
    before_drop = len(df_clean)
    df_clean = df_clean.dropna(subset=["embarked", "embark_town"])
    print(f"• 'embarked'/'embark_town' (0.22% missing < 5%): Dropped {before_drop - len(df_clean)} rows.")

    # Age: 19.87% missing (5%-30% threshold -> median imputation)
    median_age = df_clean["age"].median()
    df_clean["age"] = df_clean["age"].fillna(median_age)
    print(f"• 'age' (19.87% missing in 5%-30% range): Imputed missing values with median age ({median_age:.1f}).")

    return df_clean

# ==========================================
# TASK 3: UNIVARIATE ANALYSIS & OUTLIERS
# ==========================================

def univariate_analysis(df: pd.DataFrame):
    """Computes IQR outliers and central tendency metrics for Age and Fare."""
    print("\n" + "=" * 80)
    print("TASK 3: UNIVARIATE ANALYSIS (AGE & FARE)")
    print("=" * 80)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Age Hist & Boxplot
    sns.histplot(df["age"], kde=True, ax=axes[0, 0], color="skyblue")
    axes[0, 0].set_title("Age Distribution (Histogram & KDE)")
    sns.boxplot(x=df["age"], ax=axes[0, 1], color="lightblue")
    axes[0, 1].set_title("Age Boxplot")

    # Fare Hist & Boxplot
    sns.histplot(df["fare"], kde=True, ax=axes[1, 0], color="salmon")
    axes[1, 0].set_title("Fare Distribution (Histogram & KDE)")
    sns.boxplot(x=df["fare"], ax=axes[1, 1], color="lightcoral")
    axes[1, 1].set_title("Fare Boxplot")

    plt.suptitle("Univariate Analysis: Age and Fare Distributions", fontsize=14, fontweight="bold")
    plt.savefig(os.path.join(ANALYTICS_DIR, "univariate_age_fare.png"))
    plt.close(fig)

    # IQR Outlier Calculation
    def calculate_iqr_outliers(series: pd.Series, name: str):
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers = series[(series < lower_bound) | (series > upper_bound)]
        print(f"• {name}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f} | Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
        print(f"  Outliers Count: {len(outliers)} ({len(outliers)/len(series)*100:.2f}%)")
        return outliers

    print("\n--- IQR Outlier Detection ---")
    calculate_iqr_outliers(df["age"], "Age")
    calculate_iqr_outliers(df["fare"], "Fare")

    # Central Tendency for Fare
    fare_mean = df["fare"].mean()
    fare_median = df["fare"].median()
    fare_mode = df["fare"].mode()[0]

    print("\n--- Central Tendency Metrics for Fare ---")
    print(f"Mean Fare:   ${fare_mean:.2f}")
    print(f"Median Fare: ${fare_median:.2f}")
    print(f"Mode Fare:   ${fare_mode:.2f}")
    print(f"Skewness Order: Mean (${fare_mean:.2f}) > Median (${fare_median:.2f}) > Mode (${fare_mode:.2f})")
    print("Conclusion: Fare is strongly RIGHT-SKEWED with a heavy right tail.")

# ==========================================
# TASK 4: BIVARIATE BREAKDOWN & HEATMAP
# ==========================================

def bivariate_analysis(df: pd.DataFrame):
    """Computes survival rates across demographics and restricted 6x6 correlation matrix."""
    print("\n" + "=" * 80)
    print("TASK 4: BIVARIATE SURVIVAL RATES & CORRELATION HEATMAP")
    print("=" * 80)

    # (a) Survival by Sex
    sex_survival = df.groupby("sex")["survived"].mean() * 100
    print("\n--- (a) Survival Rate by Sex ---")
    for sex, rate in sex_survival.items():
        print(f"  Sex={sex:6s}: {rate:.2f}%")

    # (b) Survival by Pclass
    pclass_survival = df.groupby("pclass")["survived"].mean() * 100
    print("\n--- (b) Survival Rate by Pclass ---")
    for pclass, rate in pclass_survival.items():
        print(f"  Pclass={pclass}: {rate:.2f}%")

    # (c) Survival by Sex and Pclass combined
    sex_pclass_survival = df.groupby(["sex", "pclass"])["survived"].mean() * 100
    print("\n--- (c) Survival Rate by Sex & Pclass ---")
    formatted_table = sex_pclass_survival.unstack().apply(lambda col: col.map(lambda x: f"{x:.2f}%"))
    print(formatted_table)

    # Restricted 6x6 Correlation Matrix
    num_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
    corr_matrix = df[num_cols].corr()

    print("\n--- Restricted 6x6 Correlation Matrix ---")
    print(corr_matrix.round(3))

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title("6x6 Correlation Heatmap (Numeric Features & Target)", fontsize=12, fontweight="bold")
    plt.savefig(os.path.join(ANALYTICS_DIR, "correlation_heatmap.png"))
    plt.close(fig)

    # Identify top two strongest off-diagonal absolute correlations
    unstack_corr = corr_matrix.abs().unstack()
    off_diag = unstack_corr[unstack_corr < 0.9999].sort_values(ascending=False)

    top_pairs = []
    seen = set()
    for index, val in off_diag.items():
        pair = tuple(sorted(index))
        if pair not in seen:
            seen.add(pair)
            top_pairs.append((pair, corr_matrix.loc[pair[0], pair[1]]))
        if len(top_pairs) == 2:
            break

    print("\n--- Top 2 Strongest Off-Diagonal Absolute Correlations ---")
    for rank, (pair, r_val) in enumerate(top_pairs, 1):
        print(f"  {rank}. Pair: {pair[0]} <-> {pair[1]} | Pearson r = {r_val:+.3f} (abs={abs(r_val):.3f})")

# ==========================================
# TASK 5: MULTIVARIATE DATA STORY CHARTS
# ==========================================

def produce_data_story_charts(df: pd.DataFrame):
    """Generates 4 multivariate story-driven visualization charts."""
    print("\n" + "=" * 80)
    print("TASK 5: MULTIVARIATE DATA STORY CHARTS")
    print("=" * 80)

    # Chart 1: Survival Rate by Sex and Pclass
    fig1, ax1 = plt.subplots(figsize=(8, 5))
    sns.barplot(data=df, x="pclass", y="survived", hue="sex", palette="Blues_d", errorbar=None, ax=ax1)
    ax1.set_title("Chart 1: Survival Rate across Passenger Class and Gender", fontweight="bold")
    ax1.set_ylabel("Survival Rate")
    ax1.set_xlabel("Passenger Class (1 = 1st, 3 = 3rd)")
    plt.savefig(os.path.join(ANALYTICS_DIR, "chart1_survival_class_sex.png"))
    plt.close(fig1)

    # Chart 2: Fare vs. Age Scatter segmented by Survival and Pclass
    fig2, ax2 = plt.subplots(figsize=(9, 6))
    sns.scatterplot(data=df, x="age", y="fare", hue="survived", style="pclass", palette={0: "red", 1: "green"}, alpha=0.7, ax=ax2)
    ax2.set_title("Chart 2: Fare vs. Age Segmented by Survival & Class", fontweight="bold")
    ax2.set_ylim(0, 300)
    plt.savefig(os.path.join(ANALYTICS_DIR, "chart2_fare_age_survival.png"))
    plt.close(fig2)

    # Chart 3: Family Size vs. Survival
    df_temp = df.copy()
    df_temp["family_size"] = df_temp["sibsp"] + df_temp["parch"] + 1
    fig3, ax3 = plt.subplots(figsize=(8, 5))
    sns.barplot(data=df_temp, x="family_size", y="survived", palette="crest", errorbar=None, ax=ax3)
    ax3.set_title("Chart 3: Survival Rate by Family Size (SibSp + Parch + 1)", fontweight="bold")
    ax3.set_xlabel("Family Size")
    ax3.set_ylabel("Survival Rate")
    plt.savefig(os.path.join(ANALYTICS_DIR, "chart3_family_size_survival.png"))
    plt.close(fig3)

    # Chart 4: Embarked Port vs Survival by Class
    fig4, ax4 = plt.subplots(figsize=(8, 5))
    sns.barplot(data=df, x="embarked", y="survived", hue="pclass", palette="viridis", errorbar=None, ax=ax4)
    ax4.set_title("Chart 4: Survival Rate by Port of Embarkation and Class", fontweight="bold")
    ax4.set_xlabel("Port of Embarkation (C=Cherbourg, Q=Queenstown, S=Southampton)")
    ax4.set_ylabel("Survival Rate")
    plt.savefig(os.path.join(ANALYTICS_DIR, "chart4_embarked_class_survival.png"))
    plt.close(fig4)

    print("Generated 4 multivariate charts in '/analytics' directory.")

# ==========================================
# TASK 6: EXPLORATORY STANDARDIZATION CHECK
# ==========================================

def exploratory_standardization_check(df: pd.DataFrame):
    """Sanity check demonstrating z-score standardization formula z = (x - mean) / std."""
    print("\n" + "=" * 80)
    print("TASK 6: EXPLORATORY STANDARDIZATION CHECK (AGE & FARE)")
    print("=" * 80)

    age_orig = df["age"]
    fare_orig = df["fare"]

    age_z = (age_orig - age_orig.mean()) / age_orig.std()
    fare_z = (fare_orig - fare_orig.mean()) / fare_orig.std()

    comparison_df = pd.DataFrame({
        "Feature": ["Age (Original)", "Age (Z-Score)", "Fare (Original)", "Fare (Z-Score)"],
        "Mean": [age_orig.mean(), age_z.mean(), fare_orig.mean(), fare_z.mean()],
        "Std Dev": [age_orig.std(), age_z.std(), fare_orig.std(), fare_z.std()],
        "Min": [age_orig.min(), age_z.min(), fare_orig.min(), fare_z.min()],
        "Max": [age_orig.max(), age_z.max(), fare_orig.max(), fare_z.max()]
    })

    print(comparison_df.round(4).to_string(index=False))
    print("\nSanity Check Confirmed: Transformed features have Mean ≈ 0.0000 and Std Dev ≈ 1.0000.")

# ==========================================
# TASK 7 & 8: PREPROCESSING PIPELINE & CLASSIFIERS
# ==========================================

def build_and_evaluate_classifiers(df: pd.DataFrame):
    """Builds preprocessing pipeline, trains 3 classifiers, renders tree, evaluates metrics."""
    print("\n" + "=" * 80)
    print("TASK 7 & 8: STRATIFIED SPLIT, PIPELINE & CLASSIFIER EVALUATION")
    print("=" * 80)

    # Feature selection
    feature_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
    X = df[feature_cols]
    y = df["survived"]

    # Stratified Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    print(f"Train Set Shape: {X_train.shape} | Test Set Shape: {X_test.shape}")
    print(f"Train Target Distribution (Survived=1): {y_train.mean()*100:.2f}%")
    print(f"Test Target Distribution  (Survived=1): {y_test.mean()*100:.2f}%")

    # Preprocessing Column Transformers
    num_features = ["age", "fare", "sibsp", "parch"]
    cat_features = ["sex", "embarked", "pclass"]

    num_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # OneHotEncoder compatibility handling
    cat_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features)
    ])

    # Classifiers to evaluate
    models = {
        "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True)
    }

    results = []
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

    for name, clf in models.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ])

        # FIT ONLY ON TRAIN
        pipeline.fit(X_train, y_train)

        # TRANSFORM & PREDICT ON TEST
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        cm = confusion_matrix(y_test, y_pred)

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "ROC-AUC": auc,
            "TN": cm[0, 0], "FP": cm[0, 1],
            "FN": cm[1, 0], "TP": cm[1, 1]
        })

        # ROC Curve Plot
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        ax_roc.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

        # Visualize Decision Tree
        if name == "Decision Tree":
            fitted_preprocessor = pipeline.named_steps["preprocessor"]
            cat_encoder = fitted_preprocessor.named_transformers_["cat"].named_steps["onehot"]
            onehot_cols = list(cat_encoder.get_feature_names_out(cat_features))
            feature_names = num_features + onehot_cols

            fig_dt, ax_dt = plt.subplots(figsize=(16, 10))
            plot_tree(
                pipeline.named_steps["classifier"],
                feature_names=feature_names,
                class_names=["Died", "Survived"],
                filled=True,
                rounded=True,
                fontsize=8,
                ax=ax_dt
            )
            ax_dt.set_title("Decision Tree Visualization (Max Depth = 4)", fontweight="bold")
            plt.savefig(os.path.join(ANALYTICS_DIR, "decision_tree_visualization.png"))
            plt.close(fig_dt)

    ax_roc.plot([0, 1], [0, 1], "k--", label="Random Chance")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("ROC Curves Comparison across Classifiers", fontweight="bold")
    ax_roc.legend()
    plt.savefig(os.path.join(ANALYTICS_DIR, "roc_curves_comparison.png"))
    plt.close(fig_roc)

    results_df = pd.DataFrame(results)
    print("\n--- Classifier Side-by-Side Performance Comparison ---")
    print(results_df[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]].round(4).to_string(index=False))

    return X_train, X_test, y_train, y_test, preprocessor

# ==========================================
# TASK 9: CLASS IMBALANCE COMPARISON
# ==========================================

def evaluate_imbalance_handling(X_train, X_test, y_train, y_test, preprocessor):
    """Compares baseline, class_weight='balanced', and SMOTE oversampling on training fold."""
    print("\n" + "=" * 80)
    print("TASK 9: CLASS IMBALANCE HANDLING COMPARISON")
    print("=" * 80)

    print(f"Train Fold Balance: Not Survived (0) = {(y_train==0).sum()}, Survived (1) = {(y_train==1).sum()}")

    imbalance_results = []

    # 1. Baseline Random Forest
    rf_base = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(random_state=42))
    ])
    rf_base.fit(X_train, y_train)
    y_pred_base = rf_base.predict(X_test)
    imbalance_results.append({
        "Variant": "Baseline (No Handling)",
        "Precision": precision_score(y_test, y_pred_base),
        "Recall": recall_score(y_test, y_pred_base),
        "F1-Score": f1_score(y_test, y_pred_base)
    })

    # 2. Class Weight Balanced
    rf_bal = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=42))
    ])
    rf_bal.fit(X_train, y_train)
    y_pred_bal = rf_bal.predict(X_test)
    imbalance_results.append({
        "Variant": "class_weight='balanced'",
        "Precision": precision_score(y_test, y_pred_bal),
        "Recall": recall_score(y_test, y_pred_bal),
        "F1-Score": f1_score(y_test, y_pred_bal)
    })

    # 3. SMOTE on Training Fold ONLY
    X_train_trans = preprocessor.fit_transform(X_train)
    X_test_trans = preprocessor.transform(X_test)

    if HAS_SMOTE:
        smote = SMOTE(random_state=42)
        X_train_smote, y_train_smote = smote.fit_resample(X_train_trans, y_train)
        rf_smote = RandomForestClassifier(random_state=42)
        rf_smote.fit(X_train_smote, y_train_smote)
        y_pred_smote = rf_smote.predict(X_test_trans)
        imbalance_results.append({
            "Variant": "SMOTE (Train Fold Only)",
            "Precision": precision_score(y_test, y_pred_smote),
            "Recall": recall_score(y_test, y_pred_smote),
            "F1-Score": f1_score(y_test, y_pred_smote)
        })
    else:
        print("Notice: 'imblearn' package not installed; SMOTE skipped in script execution.")

    imb_df = pd.DataFrame(imbalance_results)
    print(imb_df.round(4).to_string(index=False))

# ==========================================
# TASK 10: HYPERPARAMETER TUNING & OOB SCORE
# ==========================================

def tune_random_forest_with_oob(X_train, X_test, y_train, y_test, preprocessor):
    """Executes GridSearchCV over Random Forest and evaluates OOB Score."""
    print("\n" + "=" * 80)
    print("TASK 10: HYPERPARAMETER TUNING & OUT-OF-BAG (OOB) EVALUATION")
    print("=" * 80)

    rf_base = RandomForestClassifier(oob_score=True, random_state=42, bootstrap=True)

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("clf", rf_base)
    ])

    param_grid = {
        "clf__n_estimators": [50, 100, 200],
        "clf__max_depth": [4, 6, 8, None],
        "clf__max_features": ["sqrt", "log2"]
    }

    grid_search = GridSearchCV(
        pipeline, param_grid, cv=5, scoring="f1", n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_pipeline = grid_search.best_estimator_
    best_rf = best_pipeline.named_steps["clf"]

    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Best CV F1-Score:      {grid_search.best_score_:.4f}")
    print(f"Random Forest OOB Score: {best_rf.oob_score_:.4f}")

    # Evaluate tuned model on test set
    y_pred_tuned = best_pipeline.predict(X_test)
    print(f"Tuned Test Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}")
    print(f"Tuned Test F1-Score:  {f1_score(y_test, y_pred_tuned):.4f}")

    return best_pipeline

# ==========================================
# TASK 11: REGRESSION SIDE-TASK & HETEROSCEDASTICITY
# ==========================================

def fare_regression_side_task(df: pd.DataFrame):
    """Predicts fare using multivariate linear regression & tests heteroscedasticity."""
    print("\n" + "=" * 80)
    print("TASK 11: MULTIVARIATE LINEAR REGRESSION SIDE-TASK (FARE PREDICTION)")
    print("=" * 80)

    reg_features = ["pclass", "sex", "age", "sibsp", "parch", "embarked"]
    X_reg = df[reg_features]
    y_reg = df["fare"]

    X_train, X_test, y_train, y_test = train_test_split(
        X_reg, y_reg, test_size=0.20, random_state=42
    )

    num_cols = ["age", "sibsp", "parch"]
    cat_cols = ["sex", "embarked", "pclass"]

    reg_preprocessor = ColumnTransformer(transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(drop="first", sparse_output=False))]), cat_cols)
    ])

    reg_pipeline = Pipeline(steps=[
        ("preprocessor", reg_preprocessor),
        ("regressor", LinearRegression())
    ])

    reg_pipeline.fit(X_train, y_train)
    y_pred = reg_pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    n = len(y_test)
    p = X_train.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    print(f"• Mean Absolute Error (MAE):     ${mae:.2f}")
    print(f"• Root Mean Squared Error (RMSE): ${rmse:.2f}")
    print(f"• R-squared (R²):               {r2:.4f}")
    print(f"• Adjusted R-squared (Adj R²):  {adj_r2:.4f}")

    # Residual Plot Analysis
    residuals = y_test - y_pred
    fig_res, ax_res = plt.subplots(figsize=(8, 5))
    ax_res.scatter(y_pred, residuals, alpha=0.6, color="purple")
    ax_res.axhline(0, color="red", linestyle="--")
    ax_res.set_xlabel("Predicted Fare ($)")
    ax_res.set_ylabel("Residuals ($)")
    ax_res.set_title("Residual Plot: Fare Multivariate Regression", fontweight="bold")
    plt.savefig(os.path.join(ANALYTICS_DIR, "fare_regression_residuals.png"))
    plt.close(fig_res)

    print("\nHeteroscedasticity Analysis:")
    print("Conclusion: Strong HETEROSCEDASTICITY is present. Residual spread expands dramatically as predicted fare increases.")

# ==========================================
# TASK 12 & 13: PIPELINE SAVE & RELOAD TEST
# ==========================================

def save_and_verify_pipeline(best_pipeline):
    """Saves complete fitted end-to-end pipeline artifact and tests reloading."""
    print("\n" + "=" * 80)
    print("TASK 12 & 13: END-TO-END PIPELINE ARTIFACT SAVE & RELOAD TEST")
    print("=" * 80)

    # Save full pipeline artifact
    joblib.dump(best_pipeline, MODEL_PATH)
    print(f"Saved complete fitted pipeline to '{MODEL_PATH}'.")

    # Reload pipeline from disk
    reloaded_pipeline = joblib.load(MODEL_PATH)
    print("Successfully reloaded fitted pipeline from disk.")

    # Infer on raw unpreprocessed sample data
    raw_sample = pd.DataFrame([{
        "pclass": 1,
        "sex": "female",
        "age": 29.0,
        "sibsp": 0,
        "parch": 0,
        "fare": 211.33,
        "embarked": "S"
    }])

    prediction = reloaded_pipeline.predict(raw_sample)[0]
    probabilities = reloaded_pipeline.predict_proba(raw_sample)[0]

    print("\n--- Reloaded Pipeline End-to-End Inference Verification ---")
    print("Raw Input Record:")
    print(raw_sample.to_dict(orient="records")[0])
    print(f"Predicted Class: {'Survived (1)' if prediction == 1 else 'Died (0)'}")
    print(f"Class Probabilities: [Died: {probabilities[0]:.4f}, Survived: {probabilities[1]:.4f}]")
    print("VERIFICATION SUCCESS: Reloaded pipeline operates end-to-end on raw input.")

def main():
    # Execute full analytics sequence
    df_raw = load_and_profile_data()
    df_clean = handle_missing_values(df_raw)
    univariate_analysis(df_clean)
    bivariate_analysis(df_clean)
    produce_data_story_charts(df_clean)
    exploratory_standardization_check(df_clean)

    X_train, X_test, y_train, y_test, preprocessor = build_and_evaluate_classifiers(df_clean)
    evaluate_imbalance_handling(X_train, X_test, y_train, y_test, preprocessor)
    best_pipeline = tune_random_forest_with_oob(X_train, X_test, y_train, y_test, preprocessor)
    fare_regression_side_task(df_clean)
    save_and_verify_pipeline(best_pipeline)

    print("\n" + "=" * 80)
    print("MODULE 2 ANALYTICS PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 80)

if __name__ == "__main__":
    main()

TASK 1: DATASET LOADING AND PROFILING
Fetching dataset via seaborn loader 'sns.load_dataset("titanic")'...
Saved local offline copy to '/content/titanic.csv'.

Dataset Shape: 891 rows, 15 columns

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    obj